# Day 7 — Online lab (120+ cells)

**Colab:** File → Upload this notebook → Run top to bottom.

**Rule:** Each code cell is 2–4 lines. Read the `#` comment on every line.

**Path:** Python → pandas → Polars → PySpark → FinLedger end-to-end.

## Part 1 — Install packages (Colab / fresh Jupyter)

In [ ]:
# Install FinLedger lab packages (skip if already installed)
!pip install -q pandas polars pyspark

In [ ]:
# Confirm install finished
print("Step 1 done: packages installed")

## Part 2 — Python: variables and types

In [ ]:
# A string holds text — here a transaction id
txn_id = "TXN-10003"
print("txn_id:", txn_id)

In [ ]:
# An integer counts whole numbers
row_count = 5
print("rows in sample file:", row_count)

In [ ]:
# A float holds decimal money amounts
amount = 50000.00
print("amount GBP:", amount)

In [ ]:
# A boolean is True or False — used in filters
is_posted = True
print("is_posted:", is_posted)

In [ ]:
# type() shows the Python type of a value
print(type(txn_id), type(amount), type(is_posted))

## Part 3 — Python: lists (many rows as a list)

In [ ]:
# A list keeps ordered items — channels we accept
channels = ["wire", "card", "fps"]
print(channels)

In [ ]:
# Index 0 is the first item (wire)
print("first channel:", channels[0])

In [ ]:
# -1 is the last item
print("last channel:", channels[-1])

In [ ]:
# len() counts items in the list
print("how many channels:", len(channels))

In [ ]:
# Append adds one item to the end
channels.append("chaps")
print("after append:", channels)

In [ ]:
# List comprehension — build a new list from an old one
upper = [ch.upper() for ch in channels[:3]]
print(upper)

## Part 4 — Python: dictionaries (one row as key→value)

In [ ]:
# Dict maps column names to values — like one CSV row
txn = {"transaction_id": "TXN-10003", "amount_gbp": "50000.00", "channel": "wire"}
print(txn)

In [ ]:
# Read one field with square brackets
print("id:", txn["transaction_id"])

In [ ]:
# .get() is safe if key might be missing
print("status:", txn.get("status", "unknown"))

In [ ]:
# keys() and values() list field names and values
print(list(txn.keys()))
print(list(txn.values()))

In [ ]:
# List of dicts = many rows before pandas/Spark
rows = [
    {"transaction_id": "TXN-10001", "channel": "wire", "amount_gbp": "1250.50", "status": "posted"},
    {"transaction_id": "TXN-10002", "channel": "card", "amount_gbp": "89.99", "status": "posted"},
    {"transaction_id": "TXN-10003", "channel": "wire", "amount_gbp": "50000.00", "status": "pending"},
]

In [ ]:
# How many rows in our mini dataset
print(len(rows))

## Part 5 — Python: functions (reusable rules)

In [ ]:
# Define a function — same idea as clean_amount in Session 6
def clean_amount(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0

In [ ]:
# Call the function with good and bad input
print(clean_amount("50000"))
print(clean_amount("oops"))

In [ ]:
# Loop rows and clean amount_gbp one at a time
for r in rows:
    print(r["transaction_id"], clean_amount(r["amount_gbp"]))

## Part 6 — Python: if / else (business rules)

In [ ]:
# Flag large wires for fraud monitoring
amt = clean_amount(rows[2]['amount_gbp'])
if amt >= 10000 and rows[2]["channel"] == "wire":
    print("FRAUD WATCH:", rows[2]["transaction_id"])

In [ ]:
# Keep only posted rows using a list comprehension
posted = [r for r in rows if r["status"] == "posted"]
print(len(posted), "posted rows")

## Part 7 — Python: pathlib (paths without typos)

In [ ]:
# pathlib builds paths with / instead of string concat
from pathlib import Path

In [ ]:
# Simulate lake folder layout from Session 3
run = Path("loaded") / "run=session3-lab"
print(run)

In [ ]:
# File name inside the run folder
csv_name = run / "sample_transactions.csv"
print(csv_name)

## Part 8 — pandas: load and explore (local, eager)

In [ ]:
# pandas = in-memory table on your laptop
import pandas as pd

In [ ]:
# Build DataFrame from list of dicts
pdf = pd.DataFrame(rows)
print(pdf)

In [ ]:
# .shape = (rows, columns)
print("shape:", pdf.shape)

In [ ]:
# .columns lists column names
print(list(pdf.columns))

In [ ]:
# .dtypes shows types — amount_gbp is object (string)
print(pdf.dtypes)

In [ ]:
# Select one column like a list
print(pdf["channel"])

In [ ]:
# Filter rows — only posted
posted_pdf = pdf[pdf["status"] == "posted"]
print(posted_pdf)

In [ ]:
# astype converts string column to float
pdf["amount"] = pdf["amount_gbp"].astype(float)
print(pdf[["transaction_id", "amount"]])

In [ ]:
# groupby sums amount per channel
totals = pdf.groupby("channel")["amount"].sum()
print(totals)

## Part 9 — Polars: same data, fast column engine

In [ ]:
# Polars = modern local DataFrame (Rust engine)
import polars as pl

In [ ]:
# Create Polars DataFrame from same rows
plf = pl.DataFrame(rows)
print(plf)

In [ ]:
# height = row count
print("rows:", plf.height)

In [ ]:
# select picks columns
slim = plf.select("transaction_id", "channel", "status")
print(slim)

In [ ]:
# filter keeps matching rows
posted_pl = plf.filter(pl.col("status") == "posted")
print(posted_pl)

In [ ]:
# with_columns adds typed amount column
pl_typed = plf.with_columns(pl.col("amount_gbp").cast(pl.Float64).alias("amount"))
print(pl_typed)

In [ ]:
# group_by + agg like SQL GROUP BY
by_ch = pl_typed.group_by("channel").agg(pl.col("amount").sum())
print(by_ch)

## Part 10 — Compare pandas vs Polars vs Spark (concept)

| Tool | Runs on | When to use |
|------|---------|-------------|
| pandas | Your PC RAM | Quick checks, small CSV |
| Polars | Your PC RAM | Fast local transforms |
| PySpark | Cluster / Colab CPU | Lake scale, Databricks |

In [ ]:
print("Next: PySpark — same ideas, distributed lazy engine")

## Part 11 — PySpark: SparkSession (entry point)

In [ ]:
# SparkSession is the door to read/transform/write
from pyspark.sql import SparkSession

In [ ]:
# builder creates a local Spark for Colab/Jupyter
spark = SparkSession.builder.appName("finledger-day7").master("local[*]").getOrCreate()
print("Spark version:", spark.version)

In [ ]:
# Lower log noise so output is readable
spark.sparkContext.setLogLevel("WARN")

## Part 12 — PySpark: create DataFrame (our bronze sample)

In [ ]:
# createDataFrame loads rows into a distributed lazy table
data = [
    ("TXN-10001", "wire", "1250.50", "posted"),
    ("TXN-10002", "card", "89.99", "posted"),
    ("TXN-10003", "wire", "50000.00", "pending"),
    ("TXN-10004", "card", "INVALID", "posted"),
]

In [ ]:
# Column names define the schema
cols = ["transaction_id", "channel", "amount_gbp", "status"]
df = spark.createDataFrame(data, cols)

In [ ]:
# show() is an ACTION — prints sample rows
df.show()

In [ ]:
# printSchema() shows names and types
df.printSchema()

In [ ]:
# count() is an ACTION — runs the job
print("row count:", df.count())

In [ ]:
# columns property lists names
print(df.columns)

## Part 13 — Lazy vs action (critical idea)

**Transformation (lazy):** `select`, `filter`, `withColumn` — builds a plan.

**Action (runs now):** `show`, `count`, `collect`, `write`.

In [ ]:
# filter is LAZY — no work yet
step1 = df.filter("status = 'posted'")
print("plan step added (lazy)")

In [ ]:
# count is ACTION — executes filter + count
print("posted rows:", step1.count())

In [ ]:
# show is ACTION — prints up to n rows
step1.show(5)

## Part 14 — select: pick columns

In [ ]:
# select keeps only the columns you name
slim = df.select("transaction_id", "channel", "amount_gbp", "status")

In [ ]:
# show result — still lazy until show/count
slim.show()

In [ ]:
# select with alias renames a column
from pyspark.sql import functions as F
renamed = slim.select(F.col("transaction_id").alias("txn_id"))
renamed.show()

## Part 15 — filter: keep rows

In [ ]:
# SQL string filter — easy to read
posted = df.filter("status = 'posted'")
posted.show()

In [ ]:
# Column API filter — preferred in production
posted2 = df.filter(F.col("status") == "posted")
print(posted2.count())

In [ ]:
# Combine conditions with & and |
big_wire = df.filter((F.col("channel") == "wire") & (F.col("status") == "pending"))
big_wire.show()

## Part 16 — withColumn and cast (bronze → silver typing)

In [ ]:
# Bronze CSV stores amounts as strings
df.select("amount_gbp").printSchema()

In [ ]:
# withColumn adds/replaces a column
typed = df.withColumn("amount", F.col("amount_gbp").cast("double"))
typed.printSchema()

In [ ]:
# Invalid text becomes null after cast
typed.show()

In [ ]:
# isNull() finds bad rows for quarantine (Session 9)
bad = typed.filter(F.col("amount").isNull())
bad.show()

In [ ]:
# Keep valid positive amounts
good = typed.filter((F.col("amount").isNotNull()) & (F.col("amount") > 0))
good.show()

## Part 17 — dropDuplicates (one row per transaction_id)

In [ ]:
# Add duplicate row to demo dedupe
dup_data = data + [("TXN-10001", "wire", "1250.50", "posted")]
df_dup = spark.createDataFrame(dup_data, cols)

In [ ]:
print("before dedupe:", df_dup.count())

In [ ]:
# dropDuplicates keeps one row per key
deduped = df_dup.dropDuplicates(["transaction_id"])
print("after:", deduped.count())

## Part 18 — join: enrich with lookup table

In [ ]:
# Small reference table: channel → risk level
lookup = spark.createDataFrame([("wire","high"),("card","medium"),("fps","low")], ["channel","risk"])
lookup.show()

In [ ]:
# left join keeps all transactions, adds risk
enriched = good.join(lookup, on="channel", how="left")
enriched.show()

In [ ]:
# inner join keeps only rows with a match on both sides
inner = good.join(lookup, on="channel", how="inner")
print(inner.count())

## Part 19 — groupBy and agg (gold preview)

In [ ]:
# groupBy splits rows into buckets by channel
grouped = good.groupBy("channel")

In [ ]:
# agg summarises each bucket
by_channel = grouped.agg(F.count("*").alias("txns"), F.sum("amount").alias("total_gbp"))
by_channel.show()

In [ ]:
# orderBy sorts for reporting
ranked = by_channel.orderBy(F.desc("total_gbp"))
ranked.show()

## Part 20 — window: rank within channel

In [ ]:
# Window = calculate per group without collapsing rows
from pyspark.sql.window import Window

In [ ]:
# Partition by channel, order by amount descending
w = Window.partitionBy("channel").orderBy(F.desc("amount"))

In [ ]:
# row_number assigns 1,2,3 within each channel
ranked_rows = good.withColumn("rank_in_channel", F.row_number().over(w))
ranked_rows.show()

In [ ]:
# Top 1 per channel
top1 = ranked_rows.filter(F.col("rank_in_channel") == 1)
top1.show()

## Part 21 — SQL API (same engine as DataFrame)

In [ ]:
# Register temp view to write SQL
good.createOrReplaceTempView("transactions")

In [ ]:
# SQL SELECT — learners who prefer SQL
spark.sql("SELECT channel, COUNT(*) AS n FROM transactions GROUP BY channel").show()

In [ ]:
# Filter in SQL
spark.sql("SELECT * FROM transactions WHERE amount > 1000").show()

## Part 22 — End-to-end bronze → silver → gold (in memory)

**Pipeline:** read → cast → filter good → dedupe → join → aggregate.

In [ ]:
# Step A: start from raw bronze-like df
bronze = df

In [ ]:
# Step B: cast amount to double
silver_typed = bronze.withColumn("amount", F.col("amount_gbp").cast("double"))

In [ ]:
# Step C: split good vs quarantine
silver_good = silver_typed.filter(F.col("amount").isNotNull() & (F.col("amount") > 0))
silver_bad = silver_typed.filter(F.col("amount").isNull() | (F.col("amount") <= 0))

In [ ]:
print("good:", silver_good.count(), "quarantine:", silver_bad.count())

In [ ]:
# Step D: dedupe good rows
silver = silver_good.dropDuplicates(["transaction_id"])

In [ ]:
# Step E: gold aggregate by channel
gold = silver.groupBy("channel").agg(F.sum("amount").alias("total_gbp"), F.count("*").alias("txns"))
gold.show()

## Part 23 — Read CSV from text (like a small file)

In [ ]:
# Simulate reading CSV without uploading a file
csv_text = "transaction_id,channel,amount_gbp\nTXN-20001,wire,100.00\nTXN-20002,card,50.00"

In [ ]:
# pandas read from string buffer
from io import StringIO
pdf2 = pd.read_csv(StringIO(csv_text))
print(pdf2)

## Part 24 — Common mistakes (what NOT to do)

- Do not `collect()` huge data on a cluster
- Do not paste secrets in notebooks
- Do not skip cast on bronze strings

In [ ]:
# collect() pulls ALL rows to driver — OK only for tiny data
small = good.limit(3).collect()
print(len(small), "rows on driver")

## Part 25 — Medallion paths (theory for Databricks)

On **Databricks** you will read:

```
abfss://bronze@<account>.dfs.core.windows.net/loaded/run=session3-lab/sample_transactions.csv
```

Colab cannot access your Azure lake without credentials — practice logic here, lake in Databricks `nb_04`.

## Part 26 — Python: tuples and sets

In [ ]:
# Tuple = fixed ordered pair (immutable)
pair = ("TXN-10001", "wire")
print(pair[0], pair[1])

In [ ]:
# Set = unique values only
channels_seen = {"wire", "card", "wire", "fps"}
print(channels_seen)

In [ ]:
# Set removes duplicates automatically
print(len(channels_seen), "unique channels")

## Part 27 — Python: loops with enumerate

In [ ]:
# enumerate gives index + row together
for i, r in enumerate(rows):
    print(i, r["transaction_id"])

In [ ]:
# break stops early when condition met
for r in rows:
    if r["transaction_id"] == "TXN-10003":
        print("found large wire row"); break

## Part 28 — Python: f-strings (readable messages)

In [ ]:
# f-string embeds variables inside text
msg = f"Txn {txn_id} amount {amount} GBP"
print(msg)

## Part 29 — pandas: head, describe, missing values

In [ ]:
# head() shows first n rows
print(pdf.head(2))

In [ ]:
# describe() quick numeric summary
print(pdf.describe(include="all"))

In [ ]:
# Add a row with missing status to demo nulls
pdf.loc[len(pdf)] = {"transaction_id": "TXN-99999", "channel": "wire", "amount_gbp": None, "status": None}
print(pdf.tail(1))

In [ ]:
# isna() finds missing cells
print(pdf["status"].isna())

In [ ]:
# dropna removes rows with missing values
clean_pdf = pdf.dropna(subset=["status"])
print(len(clean_pdf))

## Part 30 — Polars: more column expressions

In [ ]:
# pl.col references a column in Polars
import polars as pl

In [ ]:
# with_columns can add multiple columns at once
pl2 = plf.with_columns(pl.col("channel").str.to_uppercase().alias("channel_up"))
print(pl2)

In [ ]:
# sort by amount_gbp as string (bronze order wrong for math)
print(plf.sort("amount_gbp"))

## Part 31 — PySpark: distinct and limit

In [ ]:
# distinct() removes duplicate rows (all columns)
print(df.distinct().count())

In [ ]:
# limit(n) keeps first n rows after shuffle (use with care)
df.limit(2).show()

## Part 32 — PySpark: orderBy ascending and descending

In [ ]:
# orderBy channel A→Z
df.orderBy("channel").show()

In [ ]:
# orderBy amount_gbp descending needs cast first
tmp = df.withColumn("amount", F.col("amount_gbp").cast("double"))
tmp.orderBy(F.desc("amount")).show()

## Part 33 — PySpark: when/otherwise (conditional column)

In [ ]:
# when/otherwise = if/else as a new column
flagged = good.withColumn("fraud_flag", F.when(F.col("amount") >= 10000, "WATCH").otherwise("OK"))
flagged.show()

## Part 34 — PySpark: coalesce (first non-null)

In [ ]:
# coalesce picks first non-null column value
with_backup = df.withColumn("amt", F.coalesce(F.col("amount_gbp"), F.lit("0")))
with_backup.show()

## Part 35 — PySpark: lit (constant column)

In [ ]:
# lit injects a constant value
tagged = df.withColumn("source_system", F.lit("finledger"))
tagged.select("transaction_id", "source_system").show()

## Part 36 — PySpark: union two DataFrames

In [ ]:
# union stacks rows (same columns required)
extra = spark.createDataFrame([("TXN-20001", "fps", "10.00", "posted")], cols)
combined = df.union(extra)
print(combined.count())

## Part 37 — PySpark: cache (reuse in many actions)

In [ ]:
# cache() stores in memory after first action
cached = good.cache()

In [ ]:
# First count computes and caches
print(cached.count())

In [ ]:
# Second count is faster (reuses cache)
print(cached.count())

In [ ]:
# unpersist frees memory when done
cached.unpersist()

## Part 38 — PySpark: explain (see the plan)

In [ ]:
# explain() prints logical plan (debugging)
good.filter(F.col("amount") > 100).explain()

## Part 39 — More filters on FinLedger sample

In [ ]:
# IN list — channel is wire or card
df.filter(F.col("channel").isin(["wire", "card"])).show()

In [ ]:
# LIKE pattern — ids starting with TXN-100
df.filter(F.col("transaction_id").like("TXN-100%")).show()

In [ ]:
# NOT equal
df.filter(F.col("status") != "posted").show()

## Part 40 — Aggregations: avg, min, max

In [ ]:
# Multiple agg functions in one groupBy
stats = good.groupBy("channel").agg(
    F.avg("amount").alias("avg_gbp"),
    F.min("amount").alias("min_gbp"),
    F.max("amount").alias("max_gbp"),
)
stats.show()

## Part 41 — Window: sum running total (concept)

In [ ]:
# Unbounded window — running sum ordered by amount
w2 = Window.partitionBy("channel").orderBy("amount").rowsBetween(Window.unboundedPreceding, Window.currentRow)

In [ ]:
running = good.withColumn("running_total", F.sum("amount").over(w2))
running.show()

## Part 42 — Join types side by side

In [ ]:
# right join — all lookup rows
good.join(lookup, "channel", "right").show()

In [ ]:
# full outer — everything from both
good.join(lookup, "channel", "full").show()

## Part 43 — Silver quarantine pattern (Session 9 preview)

**FinLedger rule:** bad rows go to quarantine, pipeline keeps running.

In [ ]:
# Rebuild typed from full bronze df
typed_full = df.withColumn("amount", F.col("amount_gbp").cast("double"))

In [ ]:
silver_ok = typed_full.filter(F.col("amount").isNotNull() & (F.col("amount") > 0))
silver_bad = typed_full.filter(F.col("amount").isNull() | (F.col("amount") <= 0))

In [ ]:
print("silver_ok:", silver_ok.count(), "| quarantine:", silver_bad.count())

In [ ]:
# Show quarantine row (INVALID amount)
silver_bad.show()

## Part 44 — Gold channel summary (Session 14 preview)

In [ ]:
# Gold = business metrics for reports
gold2 = silver_ok.groupBy("channel").agg(F.sum("amount").alias("total_gbp"), F.count("*").alias("txns"))

In [ ]:
gold2.orderBy(F.desc("total_gbp")).show()

## Part 45 — Simulate pipeline parameters (widgets later)

In [ ]:
# run_id is a folder name in the lake — like a widget value
run_id = "session3-lab"
print(run_id)

In [ ]:
# Build path string (Databricks uses abfss://)
bronze_path = f"abfss://bronze@account/loaded/run={run_id}/sample_transactions.csv"
print(bronze_path)

## Part 46 — pandas → Spark handoff concept

In [ ]:
# Sometimes convert pandas → Spark (small data only)
pdf_small = pd.DataFrame(rows)
spark_from_pandas = spark.createDataFrame(pdf_small)
spark_from_pandas.show()

## Part 47 — Check: posted count all three tools

In [ ]:
# pandas count
print("pandas posted:", len(pdf[pdf["status"]=="posted"]))

In [ ]:
# Polars count
print("polars posted:", plf.filter(pl.col("status")=="posted").height)

In [ ]:
# Spark count
print("spark posted:", df.filter("status = 'posted'").count())

## Part 48 — Review: transformation chain one line at a time

In [ ]:
# One lazy step
s1 = df

In [ ]:
s2 = s1.filter("status = 'posted'")

In [ ]:
s3 = s2.withColumn("amount", F.col("amount_gbp").cast("double"))

In [ ]:
s4 = s3.filter(F.col("amount") > 0)

In [ ]:
s5 = s4.select("transaction_id", "channel", "amount")

In [ ]:
# Single action at end
s5.show()

## Part 49 — Logging vs print (production habit)

In [ ]:
# logging survives in cluster logs; print is for demos
import logging
logging.basicConfig(level=logging.INFO)
log = logging.getLogger("finledger")

In [ ]:
log.info("processing %s rows", df.count())

## Part 50 — dataclass config (Session 6 pattern)

In [ ]:
# dataclass holds run settings in one object
from dataclasses import dataclass

In [ ]:
@dataclass
class RunConfig:
    run_id: str
    storage_account: str = "stexample"

In [ ]:
cfg = RunConfig(run_id="session3-lab")
print(cfg)

## Part 51 — Final end-to-end script in tiny steps

Run these 8 cells in order — full bronze→gold logic.

In [ ]:
# 1 read bronze-like data
b = df

In [ ]:
# 2 cast
s = b.withColumn("amount", F.col("amount_gbp").cast("double"))

In [ ]:
# 3 good rows
g = s.filter(F.col("amount").isNotNull() & (F.col("amount") > 0))

In [ ]:
# 4 quarantine
q = s.filter(F.col("amount").isNull() | (F.col("amount") <= 0))

In [ ]:
# 5 dedupe
g2 = g.dropDuplicates(["transaction_id"])

In [ ]:
# 6 enrich
g3 = g2.join(lookup, "channel", "left")

In [ ]:
# 7 gold
gold_final = g3.groupBy("channel").agg(F.sum("amount").alias("total"))

In [ ]:
# 8 show gold
gold_final.show()
print("quarantine count:", q.count())

## Part 52 — Cleanup

In [ ]:
# Stop Spark to free memory
spark.stop()
print("Spark stopped")

### Checkpoint
- [ ] I ran every cell in order
- [ ] I can explain lazy vs action
- [ ] I can name select, filter, withColumn, join, groupBy, window
- [ ] Next: Databricks `nb_04_read_bronze.py` for real `abfss://`